* 작업: OD·STAY 집계와 시장 자료 결합
* 출력: 당시 저장된 실행 결과
* 재실행: 비공개 원자료 필요

In [ ]:
import zipfile
import tempfile
import shutil

import pandas as pd
import numpy as np
from io import BytesIO
from collections import defaultdict
import matplotlib.pyplot as plt
import os
import re
import time

In [ ]:
df_og = pd.read_csv("../data/market_df_merged.csv", encoding='utf-8-sig')
df_og = df_og.drop(columns = 'Unnamed: 0', errors = 'ignore')

In [ ]:
df_og.head()

In [ ]:
HOLIDAY_DATE = set([
    20230902, 20230903, 20230909, 20230910, 20230916, 20230917, 20230923,
    20230924, 20230928, 20230929, 20230930, 20231001, 20231002, 20231003,
    20231007, 20231008, 20231009, 20231014, 20231015
])

In [ ]:
AGE_MAP = {
    0: 0,
    1: 10,
    2: 20,
    3: 30,
    4: 40,
    5: 50,
    6: 60,
    7: 70,
    8: 80
}

def age_bucket(p):
    try:
        p_int = int(p)
    except Exception:
        return "age_unknown"
    return AGE_MAP.get(p_int, "age_unknown")



# purpose 그룹
PURPOSE_MAP = {
    0: "귀가",
    1: "업무",
    2: "학업",
    3: "쇼핑여가",
    4: "기타",
    5: "여행",
}

def purpose_bucket(p):
    try:
        p_int = int(p)
    except Exception:
        return "purpose_unknown"
    return PURPOSE_MAP.get(p_int, "purpose_unknown")

# holiday 그룹
def day_bucket(d):
    return "holiday" if d in HOLIDAY_DATE else "weekday"


In [ ]:
def iter_inner_csv_members_tempfile(outer_zip_path, inner_zip_name, csv_prefix=None):
    # od_20230901_10.zip 등을 임시파일로 저장한 뒤 내부 CSV member 이름을 yield
    with zipfile.ZipFile(outer_zip_path, "r") as outer:
        with outer.open(inner_zip_name) as src:
            fd, tmp_zip_path = tempfile.mkstemp(suffix=".zip")
            os.close(fd)
            try:
                with open(tmp_zip_path, "wb") as f:
                    shutil.copyfileobj(src, f, length=1024 * 1024)  # 1MB 단위 복사

                with zipfile.ZipFile(tmp_zip_path, "r") as inner:
                    members = [m for m in inner.namelist() if m.endswith(".csv")]
                    if csv_prefix is not None:
                        members = [m for m in members if os.path.basename(m).startswith(csv_prefix)]
                    for m in sorted(members):
                        yield tmp_zip_path, m  # tmp_zip_path는 나중에 inner를 다시 열기 위해 반환
            finally:
                if os.path.exists(tmp_zip_path):
                    os.remove(tmp_zip_path)

In [ ]:
def export_od_population_by_code(
    outer_zip_path,
    code,
    inner_zip_names,
    out_dir,
    usecols=None,
    chunksize=500_000
):
    # 1개의 행정동씩 dest_hdong_cd==code만 필터링해서 저장(append)
    code = str(code)
    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, f"od_population_{code}.csv")

    if usecols is None:
        usecols = ["origin_hdong_cd","dest_hdong_cd","date","age","dest_purpose","modal","od_cnts"]

    first_write = True

    for inner_zip_name in inner_zip_names:
        # inner zip 안의 csv members 순회
        for tmp_zip_path, csv_member in iter_inner_csv_members_tempfile(
            outer_zip_path, inner_zip_name, csv_prefix="od_"
        ):
            with zipfile.ZipFile(tmp_zip_path, "r") as inner:
                with inner.open(csv_member) as f:
                    for chunk in pd.read_csv(f, usecols=usecols, chunksize=chunksize):
                        chunk["dest_hdong_cd"] = chunk["dest_hdong_cd"].astype(str)
                        chunk = chunk[chunk["dest_hdong_cd"] == code]
                        if chunk.empty:
                            continue

                        chunk.to_csv(out_path, index=False, mode="w" if first_write else "a",
                                     header=first_write, encoding="utf-8-sig")
                        first_write = False

    return out_path


In [ ]:
outer_zip_path = "../data/raw/데이터분석 분야_로우데이터.zip"

# market_df 준비
market_df = df_og.copy()
market_df["행정기관코드"] = market_df["행정기관코드"].astype(str)
code_list = market_df["행정기관코드"].astype(str).unique().tolist()

OD_INNER_ZIPS = [
    "od_20230901_10.zip",
    "od_20230911_20.zip",
    "od_20230921_30.zip",
    "od_20231001_15.zip",
]
STAY_INNER_ZIPS = [
    "stay_20230901_15.zip",
    "stay_20230916_30.zip",
    "stay_20231001_15.zip",
]

## OD 집계

In [ ]:
def extract_od_features(
    outer_zip_path,
    target_codes,
    inner_zip_names,
    chunksize=500_000
):
    
    target_set = set(map(str, target_codes))

    usecols = ["dest_hdong_cd","date","age","dest_purpose","modal","od_cnts"]

    # acc[code][feature] = value 누적
    acc = defaultdict(lambda: defaultdict(float))

    for inner_zip_name in inner_zip_names:
        for tmp_zip_path, csv_member in iter_inner_csv_members_tempfile(
            outer_zip_path, inner_zip_name, csv_prefix="od_"
        ):
            with zipfile.ZipFile(tmp_zip_path, "r") as inner:
                with inner.open(csv_member) as f:
                    for df in pd.read_csv(f, usecols=usecols, chunksize=chunksize):
                        df["dest_hdong_cd"] = df["dest_hdong_cd"].astype(str)
                        df = df[df["dest_hdong_cd"].isin(target_set)]
                        if df.empty:
                            continue

                        # 날짜 int로 (안되면 string->int 변환)
                        if df["date"].dtype != np.int64 and df["date"].dtype != np.int32:
                            df["date"] = df["date"].astype(int, errors="ignore")

                        # total
                        g_total = df.groupby("dest_hdong_cd")["od_cnts"].sum()
                        for code, v in g_total.items():
                            acc[code]["od_in_total"] += float(v)

                        # purpose 그룹(일상/여가)
                        df["_purpose_grp"] = df["dest_purpose"].apply(purpose_bucket)
                        g_pgrp = df.groupby(["dest_hdong_cd","_purpose_grp"])["od_cnts"].sum()
                        for (code, grp), v in g_pgrp.items():
                            acc[code][f"od_in_{grp}"] += float(v)

                        # age 그룹(4개)
                        df["_age_grp"] = df["age"].apply(age_bucket)
                        g_agrp = df.groupby(["dest_hdong_cd","_age_grp"])["od_cnts"].sum()
                        for (code, grp), v in g_agrp.items():
                            acc[code][f"od_in_{grp}"] += float(v)

                        # modal별(원본 값 유지)
                        g_modal = df.groupby(["dest_hdong_cd","modal"])["od_cnts"].sum()
                        for (code, modal), v in g_modal.items():
                            acc[code][f"od_in_modal_{modal}"] += float(v)

                        # 주중/휴일
                        df["_day_grp"] = df["date"].apply(day_bucket)
                        g_day = df.groupby(["dest_hdong_cd","_day_grp"])["od_cnts"].sum()
                        for (code, grp), v in g_day.items():
                            acc[code][f"od_in_{grp}"] += float(v)

    # dict -> df
    rows = []
    for code in target_set:
        row = {"행정기관코드": code}
        row.update(acc[code])
        rows.append(row)

    return pd.DataFrame(rows).fillna(0)


In [ ]:
od_feat = extract_od_features(outer_zip_path, code_list, OD_INNER_ZIPS, chunksize=300_000)

In [ ]:
#od_feat_prev = od_feat.copy()

In [ ]:
od_feat

## STAY 집계

In [ ]:
def extract_stay_features(
    outer_zip_path,
    target_codes,
    inner_zip_names,
    chunksize=500_000
):
    target_set = set(map(str, target_codes))

    usecols = ["hdong_cd","date","time","age","purpose","stay_cnts"]

    acc = defaultdict(lambda: defaultdict(float))

    for inner_zip_name in inner_zip_names:
        for tmp_zip_path, csv_member in iter_inner_csv_members_tempfile(
            outer_zip_path, inner_zip_name, csv_prefix="stay_"
        ):
            with zipfile.ZipFile(tmp_zip_path, "r") as inner:
                with inner.open(csv_member) as f:
                    for df in pd.read_csv(f, usecols=usecols, chunksize=chunksize):
                        df["hdong_cd"] = df["hdong_cd"].astype(str)
                        df = df[df["hdong_cd"].isin(target_set)]
                        if df.empty:
                            continue

                        if df["date"].dtype != np.int64 and df["date"].dtype != np.int32:
                            df["date"] = df["date"].astype(int, errors="ignore")

                        # total
                        g_total = df.groupby("hdong_cd")["stay_cnts"].sum()
                        for code, v in g_total.items():
                            acc[code]["stay_total"] += float(v)

                        # purpose 그룹
                        df["_purpose_grp"] = df["purpose"].apply(purpose_bucket)
                        g_pgrp = df.groupby(["hdong_cd","_purpose_grp"])["stay_cnts"].sum()
                        for (code, grp), v in g_pgrp.items():
                            acc[code][f"stay_{grp}"] += float(v)

                        # age 그룹
                        df["_age_grp"] = df["age"].apply(age_bucket)
                        g_agrp = df.groupby(["hdong_cd","_age_grp"])["stay_cnts"].sum()
                        for (code, grp), v in g_agrp.items():
                            acc[code][f"stay_{grp}"] += float(v)

                        # 시간대별 체류(원본 time 유지)
                        g_time = df.groupby(["hdong_cd","time"])["stay_cnts"].sum()
                        for (code, t), v in g_time.items():
                            acc[code][f"stay_time_{t}"] += float(v)

                        # 주중/휴일
                        df["_day_grp"] = df["date"].apply(day_bucket)
                        g_day = df.groupby(["hdong_cd","_day_grp"])["stay_cnts"].sum()
                        for (code, grp), v in g_day.items():
                            acc[code][f"stay_{grp}"] += float(v)

    rows = []
    for code in target_set:
        row = {"행정기관코드": code}
        row.update(acc[code])
        rows.append(row)

    return pd.DataFrame(rows).fillna(0)


In [ ]:
# stay_feat_prev = stay_feat.copy()

In [ ]:
stay_feat = extract_stay_features(outer_zip_path, code_list, STAY_INNER_ZIPS, chunksize=300_000)

In [ ]:
stay_feat.head(5)

In [ ]:
feat = od_feat.merge(stay_feat, on="행정기관코드", how="outer").fillna(0)

In [ ]:
feat["od_in_modal_0"] = (
    feat.get("od_in_modal_0", 0)
    + feat.get("od_in_modal_0.0", 0)
)
feat.drop(columns=["od_in_modal_0.0"], inplace=True, errors="ignore")

feat["od_in_modal_1"] = (
    feat.get("od_in_modal_1", 0)
    + feat.get("od_in_modal_1.0", 0)
)
feat.drop(columns=["od_in_modal_1.0"], inplace=True, errors="ignore")

feat["od_in_modal_2"] = (
    feat.get("od_in_modal_2", 0)
    + feat.get("od_in_modal_2.0", 0)
)
feat.drop(columns=["od_in_modal_2.0"], inplace=True, errors="ignore")

feat["od_in_modal_3"] = (
    feat.get("od_in_modal_3", 0)
    + feat.get("od_in_modal_3.0", 0)
)
feat.drop(columns=["od_in_modal_3.0"], inplace=True, errors="ignore")

feat["od_in_modal_4"] = (
    feat.get("od_in_modal_4", 0)
    + feat.get("od_in_modal_4.0", 0)
)
feat.drop(columns=["od_in_modal_4.0"], inplace=True, errors="ignore")

feat["od_in_modal_5"] = (
    feat.get("od_in_modal_5", 0)
    + feat.get("od_in_modal_5.0", 0)
)
feat.drop(columns=["od_in_modal_5.0"], inplace=True, errors="ignore")

feat["od_in_modal_7"] = (
    feat.get("od_in_modal_7", 0)
    + feat.get("od_in_modal_7.0", 0)
)
feat.drop(columns=["od_in_modal_7.0"], inplace=True, errors="ignore")



In [ ]:
feat.columns

In [ ]:
sorted_cols = ['행정기관코드', 
               
               'od_in_total', 'od_in_귀가', 'od_in_기타', 'od_in_쇼핑여가',
               'od_in_업무', 'od_in_여행', 'od_in_학업', 

               #'od_in_age_adole', 'od_in_age_middle', 'od_in_age_senior', 'od_in_age_young',
               'od_in_0', 'od_in_10', 'od_in_20', 'od_in_30', 
               'od_in_40', 'od_in_50', 'od_in_60', 'od_in_70', 
               #'od_in_age_80',

               'od_in_modal_0', 'od_in_modal_1', 'od_in_modal_2', 'od_in_modal_3',
               'od_in_modal_4', 'od_in_modal_5', 'od_in_modal_7', 

               'od_in_weekday', 'od_in_holiday',
              
               'stay_total', 'stay_귀가', 'stay_기타', 'stay_쇼핑여가',
               'stay_업무', 'stay_여행', 'stay_학업', 
              
               #'stay_age_adole', 'stay_age_middle', 'stay_age_senior', 'stay_age_young', 
               'stay_0', 'stay_10', 'stay_20', 'stay_30', 'stay_40',
               'stay_50', 'stay_60', 'stay_70', 'stay_80',
              
               'stay_time_08:00', 'stay_time_09:00', 'stay_time_10:00', 
               'stay_time_11:00', 'stay_time_12:00', 'stay_time_13:00', 
               'stay_time_14:00', 'stay_time_15:00', 'stay_time_16:00', 
               'stay_time_17:00', 'stay_time_18:00', 'stay_time_19:00', 
               'stay_time_20:00', 'stay_time_21:00', 'stay_time_22:00', 
               'stay_time_23:00', 
              
               'stay_weekday','stay_holiday']

eng_cols = [  '행정기관코드', 
               
              'od_in_total', 'od_in_home', 'od_in_other', 'od_in_shop_leisure',
              'od_in_work', 'od_in_travel', 'od_in_edu', 

              #'od_in_age_adole', 'od_in_age_middle', 'od_in_age_senior', 'od_in_age_young',
              'od_in_age_0', 'od_in_age_10', 'od_in_age_20', 'od_in_age_30', 
              'od_in_age_40', 'od_in_age_50', 'od_in_age_60', 'od_in_age_70', 
              #'od_in_age_80',

              'od_in_modal_0', 'od_in_modal_1', 'od_in_modal_2', 'od_in_modal_3',
              'od_in_modal_4', 'od_in_modal_5', 'od_in_modal_7', 

              'od_in_weekday', 'od_in_holiday',

              'stay_total', 'stay_home', 'stay_other', 'stay_shop_leisure',
              'stay_work', 'stay_travel', 'stay_edu', 

              #'stay_age_adole', 'stay_age_middle', 'stay_age_senior', 'stay_age_young', 
              'stay_age_0', 'stay_age_10', 'stay_age_20', 'stay_age_30', 'stay_age_40',
              'stay_age_50', 'stay_age_60', 'stay_age_70', 'stay_age_80',

              'stay_time_08:00', 'stay_time_09:00', 'stay_time_10:00', 
              'stay_time_11:00', 'stay_time_12:00', 'stay_time_13:00', 
              'stay_time_14:00', 'stay_time_15:00', 'stay_time_16:00', 
              'stay_time_17:00', 'stay_time_18:00', 'stay_time_19:00', 
              'stay_time_20:00', 'stay_time_21:00', 'stay_time_22:00', 
              'stay_time_23:00', 

              'stay_weekday','stay_holiday']

In [ ]:
feat_eng = feat[sorted_cols].copy()
feat_eng.columns = eng_cols
od_stay_final = feat_eng[eng_cols].copy()

In [ ]:
od_stay_final

### 중간저장

In [ ]:
od_stay_final.to_csv("../data/od_stay_final.csv", encoding='utf-8-sig', index_label= False)

In [ ]:
od_stay_final.columns

## 변수처리

In [ ]:
import re
import numpy as np
import pandas as pd

def build_behavior_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # 1) 시간대
    time_cols = [c for c in df.columns if c.startswith("stay_time_")]

    def _time_to_hour(col):
        # stay_time_08:00, stay_time_8:00, stay_time_08:00:00 등 방어
        m = re.search(r"stay_time_(\d{1,2})", col)
        return int(m.group(1)) if m else None

    hour_map = {c: _time_to_hour(c) for c in time_cols}
    # 구간 정의: morning 8-11, lunch 12-15, evening 16-19, night 20-23
    buckets = {
        "stay_morning": [c for c,h in hour_map.items() if h is not None and 8  <= h <= 11],
        "stay_lunch":   [c for c,h in hour_map.items() if h is not None and 12 <= h <= 15],
        "stay_evening": [c for c,h in hour_map.items() if h is not None and 16 <= h <= 19],
        "stay_night":   [c for c,h in hour_map.items() if h is not None and 20 <= h <= 23],
    }
    for new_col, cols in buckets.items():
        if cols:
            df[new_col] = df[cols].sum(axis=1)
        else:
            df[new_col] = 0.0


    # 2) 연령 (OD/Stay)
    # adole: 0,10 / young: 20,30 / middle: 40,50 / senior: 60,70,80
    age_groups = {
        "adole": [0, 10],
        "young": [20, 30],
        "middle": [40, 50],
        "senior": [60, 70, 80],
    }

    def _sum_by_suffix(prefix, suffix_list):
        cols = [f"{prefix}{s}" for s in suffix_list if f"{prefix}{s}" in df.columns]
        return df[cols].sum(axis=1) if cols else 0.0

    # OD
    for gname, suffixes in age_groups.items():
        df[f"od_{gname}"] = _sum_by_suffix("od_in_age_", suffixes)

    # Stay
    for gname, suffixes in age_groups.items():
        df[f"stay_{gname}"] = _sum_by_suffix("stay_age_", suffixes)

    # 3) 이동수단 (OD)
    # 0:차량, 1:시내버스, 2:지하철, 3:도보, 4:기타, 5:철도, 6:시외고속버스, 7:항공기
    # bus = 1+6, sub = 2+5, etc = 4+7
    modal_groups = {
        "car":  [0],
        "walk": [3],
        "bus":  [1, 6],
        "sub":  [2, 5],
        "etc":  [4, 7],
    }

    for gname, suffixes in modal_groups.items():
        df[f"od_modal_{gname}"] = _sum_by_suffix("od_in_modal_", suffixes)


    # 4) 체류 목적
    # stay_consumption = shop_leisure + travel
    # stay_workedu     = work + edu
    # stay_residential = home + other
    purpose_groups = {
        "consumption": ["shop_leisure", "travel"],
        "workedu":     ["work", "edu"],
        "residential": ["home", "other"],
    }

    # OD
    for grp, suffixes in purpose_groups.items():
        cols = [f"od_in_{s}" for s in suffixes if f"od_in_{s}" in df.columns]
        df[f"od_{grp}"] = df[cols].sum(axis=1) if cols else 0.0

    # Stay
    for grp, suffixes in purpose_groups.items():
        cols = [f"stay_{s}" for s in suffixes if f"stay_{s}" in df.columns]
        df[f"stay_{grp}"] = df[cols].sum(axis=1) if cols else 0.0

    return df

# 사용 예시
# df = pd.read_csv("/mnt/data/market_od.csv")
# df = build_behavior_features(df)


In [ ]:
od_stay_final_merged = build_behavior_features(od_stay_final)

In [ ]:
od_stay_final_merged.to_csv("../data/od_stay_final_merged.csv", encoding='utf-8-sig', index_label=False)

## market 데이터와 병합

In [ ]:
import pandas as pd
od_stay_final_merged = pd.read_csv("../data/od_stay_final_merged.csv", encoding='utf-8-sig')
# od_stay_final = od_stay_final.drop(columns = 'Unnamed: 0',)
df_og = pd.read_csv("../data/market_df_merged.csv", encoding='utf-8-sig')
#df_og = df_og.drop(columns = 'Unnamed: 0', errors = 'ignore')


In [ ]:
col_lists = [
    # 연령대 유입: 4개
    "od_adole", "od_young", "od_middle", "od_senior",

    # 유입목적: 3개
    'od_consumption', 'od_workedu',	'od_residential',

    # 주중/주말 유입: 2개
    'od_in_weekday', 'od_in_holiday', 
    
    # 유입 시 사용한 이동수단: 5개
    "od_modal_car", "od_modal_walk", "od_modal_bus", "od_modal_sub", "od_modal_etc",
    


    # 연령대 체류: 4개
    "stay_adole", "stay_young", "stay_middle", "stay_senior",

    # 체류목적: 3개
    'stay_consumption', 'stay_workedu',	'stay_residential',

    # 주중/주말 체류: 2개
    'stay_weekday', 'stay_holiday',

    # 시간대 체류: 4개
    'stay_morning', 'stay_lunch', 'stay_evening', 'stay_night'
]

In [ ]:
od_stay_final_merged['행정기관코드'] = od_stay_final_merged['행정기관코드'].astype(str)
df_og['행정기관코드'] = df_og['행정기관코드'].astype(str)


market_od = df_og.merge(
    od_stay_final_merged[['행정기관코드'] + col_lists],
    on='행정기관코드',
    how='left'
)

In [ ]:
market_od.head(3)

In [ ]:
market_od.columns

## 저장

In [ ]:
market_od.to_csv("../data/market_od.csv", encoding='utf-8-sig', index_label=False)